# Сборщик мусора в Python

# 1. Объекты, имена и ссылки

В Python переменная не хранит объект «внутри себя». Она связывает имя с объектом в памяти. Несколько имён могут указывать на один и тот же объект.

<figure style="text-align: center; margin: 1.5em;">
  <img src="images_garbage_collector_python/names_objects.png" alt="Два имени ссылаются на один объект списка в памяти" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 1. Идентичность (is) и равенство (==): a и b — один объект, c — равный, но другой
  </figcaption>
</figure>

Оператор `is` проверяет идентичность объектов, а `==` — равенство их значений.

In [1]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]

print('a is b:', a is b)
print('a == b:', a == b)
print('a is c:', a is c)
print('a == c:', a == c)
print('id(a), id(b), id(c):', id(a), id(b), id(c))

a is b: True
a == b: True
a is c: False
a == c: True
id(a), id(b), id(c): 139568691910656 139568691910656 139568691912256


## Где возникают ссылки

Объект могут удерживать не только локальные переменные. Ссылки бывают в атрибутах экземпляра, элементах контейнеров, глобальных переменных, замыканиях, очередях, кэшах и callback-функциях. Именно поэтому объект иногда «не удаляется», хотя имя, с которым работал программист, уже удалено.

<figure style="text-align: center; margin: 1.5em;">
  <img src="images_garbage_collector_python/hidden_refs.png" alt="Объект остаётся в памяти, пока на него есть хотя бы одна ссылка" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 2. Объект удерживают не только локальные переменные
  </figcaption>
</figure>

# 2. Подсчёт ссылок: основной механизм CPython

У объекта CPython есть число сильных ссылок — *reference count*. При появлении сильной ссылки оно увеличивается, при исчезновении — уменьшается. Когда счётчик достигает нуля, объект обычно уничтожается немедленно.

Это быстро и предсказуемо, но имеет принципиальное ограничение: механизм видит число ссылок, а не достижимость объекта из работающей программы.

In [2]:
import sys

obj = []
print('После создания:', sys.getrefcount(obj))
alias = obj
print('После alias = obj:', sys.getrefcount(obj))
del alias
print('После del alias:', sys.getrefcount(obj))

# Важно: getrefcount(obj) сам временно передаёт ссылку в функцию,
# поэтому результат обычно на 1 больше интуитивно ожидаемого.

После создания: 2
После alias = obj: 3
После del alias: 2


Момент уничтожения объекта удобно увидеть через `__del__`: CPython вызывает его, когда счётчик ссылок достигает нуля. Здесь `__del__` используется только для демонстрации (см. раздел 7).

Обратите внимание: `del` удаляет **имя**, а не объект. Объект уничтожается, только когда не остаётся ни одной ссылки.

In [3]:
class Resource:
    def __init__(self, name):
        self.name = name
        print(f'  создан {self.name}')

    def __del__(self):
        print(f'  уничтожен {self.name}')


print('1. del удаляет имя; объект живёт, пока есть другая ссылка')
first = Resource('A')
second = first
del first
print('  имя first удалено, но объект ещё жив')
del second
print('  после del second объект уже уничтожен\n')

print('2. Выход из функции')
def work():
    temp = Resource('B')
    print('  внутри функции')

work()
print('  функция завершилась\n')

print('3. Замена ссылки новым значением')
item = Resource('C')
item = Resource('D')
print('  C уничтожен сразу при переприсваивании')
del item

1. del удаляет имя; объект живёт, пока есть другая ссылка
  создан A
  имя first удалено, но объект ещё жив
  уничтожен A
  после del second объект уже уничтожен

2. Выход из функции
  создан B
  внутри функции
  уничтожен B
  функция завершилась

3. Замена ссылки новым значением
  создан C
  создан D
  уничтожен C
  C уничтожен сразу при переприсваивании
  уничтожен D


# 3. Проблема циклических ссылок

Рассмотрим два списка, которые содержат друг друга. После удаления внешних имён объекты уже недоступны программе, но каждый по-прежнему имеет ссылку от другого. Их счётчики не станут нулевыми.

<figure style="text-align: center; margin: 1.5em;">
  <img src="images_garbage_collector_python/cyclic_refs.png" alt="Два списка, ссылающиеся друг на друга, остаются в памяти после удаления имён" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 3. Циклическая ссылка: после удаления имён счётчики остаются ненулевыми
  </figcaption>
</figure>

Это **недостижимый объект**, но reference counting самостоятельно освободить его не может.

In [4]:
import gc

gc.collect()  # очищаем следы предыдущих экспериментов
a = []
b = []
a.append(b)
b.append(a)

del a, b
collected = gc.collect()
print(f'gc.collect() нашёл недостижимых объектов: {collected}')

# Число зависит от окружения и от того, какие объекты создаёт сам Jupyter.

gc.collect() нашёл недостижимых объектов: 2


# 4. Циклический garbage collector

В CPython cyclic GC — дополнение к подсчёту ссылок. Он работает с отслеживаемыми объектами, прежде всего контейнерами и экземплярами классов, которые потенциально могут образовать цикл.

Упрощённая модель работы:

1. GC рассматривает кандидатов как граф: объекты — вершины, ссылки — рёбра.
2. Он учитывает внутренние ссылки между кандидатами.
3. Объекты, достижимые из корней программы (например, стеков и глобальных переменных), остаются живыми.
4. Изолированные недостижимые компоненты графа признаются мусором и освобождаются.

Иными словами: reference counting удаляет объекты с нулём ссылок, а cyclic GC удаляет **недостижимые циклы**.

<figure style="text-align: center; margin: 1.5em;">
  <img src="images_garbage_collector_python/cyclic_gc_steps_dark.png" alt="Циклический GC вычитает внутренние ссылки из копий счётчиков и освобождает недостижимый цикл" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 4. Циклический GC: вычитание внутренних ссылок и освобождение недостижимого цикла
  </figcaption>
</figure>

In [5]:
samples = [42, 'строка', (), [], {}, {'x': 1}, {'x': []}]
for item in samples:
    print(f'{item!r:15} → tracked by GC: {gc.is_tracked(item)}')

# Отслеживаются контейнеры, которые потенциально могут участвовать в цикле.
# Пустой словарь и словарь только с простыми значениями пока не отслеживаются.
# Само отслеживание — деталь реализации CPython, а не обещание языка.

42              → tracked by GC: False
'строка'        → tracked by GC: False
()              → tracked by GC: False
[]              → tracked by GC: True
{}              → tracked by GC: True
{'x': 1}        → tracked by GC: True
{'x': []}       → tracked by GC: True


# 5. Поколения

Большинство объектов живёт недолго: временные списки, строки, результаты вычислений. Поэтому GC применяет гипотезу поколений и не проверяет все старые объекты при каждой небольшой аллокации.

<figure style="text-align: center; margin: 1.5em;">
  <img src="images_garbage_collector_python/gc_generations.png" alt="Три поколения сборщика мусора: объекты, переживая сборки, переходят в старшие поколения" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 5. Поколения GC: чем старше объект, тем реже его проверяют
  </figcaption>
</figure>

Объекты, пережившие сборку, перемещаются в более старшее поколение. Детали порогов и поведения менялись между версиями Python, поэтому в отчёте обязательно указывают версию интерпретатора.

In [6]:
import pprint

print('Версия Python:', sys.version.split()[0])
print('Автоматический GC включён:', gc.isenabled())
print('Текущие счётчики:', gc.get_count())
print('Пороги:', gc.get_threshold())
print('Статистика:')
pprint.pp(gc.get_stats())

#   collections   — сколько раз запускалась сборка этого поколения;
#   collected     — сколько недостижимых объектов найдено и освобождено;
#   uncollectable — сколько объектов GC нашёл, но освободить не смог
#                   (они попадают в gc.garbage). Обычно 0: начиная с
#                   Python 3.4 (PEP 442) даже циклы с __del__ собираются.
#                   Ненулевое значение — сигнал проблемы, чаще всего
#                   в старом C-расширении.

Версия Python: 3.14.7
Автоматический GC включён: True
Текущие счётчики: (333, 0, 0)
Пороги: (2000, 10, 10)
Статистика:
[{'collections': 49, 'collected': 807, 'uncollectable': 0},
 {'collections': 4, 'collected': 18, 'uncollectable': 0},
 {'collections': 2, 'collected': 553, 'uncollectable': 0}]


## Полная диаграмма работы GC

<figure style="text-align: center; margin: 1.5em;">
  <img src="images_garbage_collector_python/gc_diagramm.png" alt="Диаграмма работы Garbage Collector" style="width: 85%; max-width: 900px;">
  <figcaption style="font-size: 0.9em; color: gray; margin-top: 0.5em;">
    Рис. 6. Диаграмма работы Garbage Collector
  </figcaption>
</figure>

## Наблюдение за автоматическими сборками

`gc.callbacks` позволяет подключить функцию, которую CPython вызовет до и после каждой сборки. Ниже мы считаем число событий; это безопаснее, чем печатать сообщение на каждый запуск GC.

In [9]:
class Node:
    pass

events = []

def gc_observer(phase, info):
    if phase == 'stop':
        events.append((info['generation'], info['collected'], info['uncollectable']))

gc.callbacks.append(gc_observer)
try:
    for _ in range(5_000):
        n1, n2 = Node(), Node()
        n1.other, n2.other = n2, n1   # цикл, который сам не освободится
finally:
    gc.callbacks.remove(gc_observer)

print('(поколение, собрано, несобираемых):', events)

(поколение, собрано, несобираемых): [(0, 1974, 0), (1, 2050, 0), (0, 2016, 0), (0, 1998, 0), (0, 1996, 0)]


# 6. `gc`: главные функции

| Функция | Назначение |
|---|---|
| `gc.isenabled()` | Проверить, включена ли автоматическая сборка. |
| `gc.enable()` / `gc.disable()` | Включить или отключить автоматический cyclic GC. |
| `gc.collect(generation=2)` | Запустить сборку вручную. Без аргумента — полную (поколение 2); `0` или `1` проверяют только младшие поколения. Возвращает число найденных недостижимых объектов. |
| `gc.get_count()` | Получить текущие счётчики для поколений. |
| `gc.get_threshold()` / `gc.set_threshold(t0, t1, t2)` | Получить или изменить пороги запуска сборки для каждого поколения. |
| `gc.get_stats()` | Получить статистику сборок. |
| `gc.get_objects()` | Список всех объектов, отслеживаемых GC. Может быть очень большим; только для отладки. |
| `gc.get_referrers(obj)` | Исследовать, кто удерживает объект; только для отладки. |
| `gc.freeze()` / `gc.unfreeze()` | Перенести все текущие объекты в «постоянное» поколение, которое GC больше не проверяет. Применяют перед `fork`, например, при предзагрузке приложения в gunicorn: меньше страниц памяти копируется в дочерних процессах. |
| `gc.garbage` | Список объектов, которые GC нашёл, но не смог освободить. Обычно пуст. |
| `gc.set_debug(flags)` | Включить отладочный вывод GC. Флаги: `DEBUG_STATS`, `DEBUG_COLLECTABLE`, `DEBUG_UNCOLLECTABLE`, `DEBUG_SAVEALL`, `DEBUG_LEAK`. |

Не стоит вызывать `gc.collect()` в горячем цикле или после каждого веб-запроса «на всякий случай»: полная проверка может замедлить программу. Отключение GC оправдано только после измерений и анализа ссылок.

# 7. Финализация, `__del__` и внешние ресурсы

Память Python-объекта и внешний ресурс — не одно и то же. Файл, сетевое соединение или блокировка должны освобождаться в определённый момент, а не когда-нибудь при сборке мусора. Поэтому для них используют контекстный менеджер.

Метод `__del__` не следует использовать как основной механизм освобождения ресурсов: время финализации может быть неудобным для программы, а поведение с циклами и исключениями усложняет отладку.

In [10]:
from pathlib import Path

demo_file = Path('gc_demo.txt')
demo_file.write_text('Ресурс будет закрыт при выходе из блока.\n', encoding='utf-8')

with demo_file.open(encoding='utf-8') as file:
    print(file.read().strip())

demo_file.unlink()
print('Файл удалён после безопасного закрытия.')

Ресурс будет закрыт при выходе из блока.
Файл удалён после безопасного закрытия.


С Python 3.4 (PEP 442) объекты с `__del__` внутри циклов собираются корректно, поэтому «`__del__` ломает GC» — устаревший совет. Но `__del__` по-прежнему не гарантирует момент вызова и плохо подходит для освобождения внешних ресурсов.

# 8. Слабые ссылки

Сильная ссылка удерживает объект живым. Слабая ссылка (`weakref`) позволяет наблюдать за объектом, но не продлевает его жизнь. Это полезно для кэшей, реестров объектов и паттерна observer. Слабую ссылку нельзя создать на любой объект: встроенные `list`, `dict`, `int`, `str` и `tuple` её не поддерживают (будет `TypeError`). Обычно достаточно обернуть данные в собственный класс или создать подкласс, например `class MyList(list): pass`.

In [11]:
import weakref

class User:
    pass

user = User()
reference = weakref.ref(user)
print('Слабая ссылка указывает на объект:', reference() is user)

del user
gc.collect()
print('После удаления последней сильной ссылки:', reference())

Слабая ссылка указывает на объект: True
После удаления последней сильной ссылки: None


In [12]:
try:
    weakref.ref([])
except TypeError as error:
    print('Ошибка:', error)  # cannot create weak reference to 'list' object

Ошибка: cannot create weak reference to 'list' object


# 9. Утечка памяти: что ею является, а что — нет

Рост памяти процесса не всегда означает утечку: аллокатор CPython может оставить освобождённые блоки для последующего повторного использования. Цикл ссылок также не всегда утечка — его может собрать cyclic GC.

Настоящая утечка в прикладном Python-коде часто означает, что объект **всё ещё достижим**: его хранит глобальный список, бесконечный кэш, очередь, замыкание, подписка на событие или callback.

Для вопроса «какие строки выделили память?» подходит `tracemalloc`. Для вопроса «кто удерживает конкретный объект?» — `gc.get_referrers()` (осторожно: только диагностика) или специализированный `objgraph`.

In [13]:
import tracemalloc

tracemalloc.start()
payload = [list(range(1_000)) for _ in range(100)]
snapshot = tracemalloc.take_snapshot()

print('Топ-5 источников выделений:')
for stat in snapshot.statistics('lineno')[:5]:
    print(stat)

del payload
tracemalloc.stop()

Топ-5 источников выделений:
/tmp/ipykernel_269078/3905037212.py:4: size=3107 KiB, count=74412, average=43 B
/home/defenser/miniconda3/lib/python3.14/codeop.py:117: size=710 B, count=9, average=79 B
/home/defenser/miniconda3/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3762: size=312 B, count=1, average=312 B
/home/defenser/miniconda3/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: size=168 B, count=1, average=168 B
/home/defenser/miniconda3/lib/python3.14/site-packages/traitlets/traitlets.py:749: size=131 B, count=2, average=66 B


In [14]:

_cache = []  # глобальный список «на всякий случай»

class Report:
    pass

def alive():
    return sum(isinstance(o, Report) for o in gc.get_objects())

for _ in range(100):
    _cache.append(Report())

gc.collect()
print('Живых Report:', alive())
print('Кто держит:', [type(h).__name__ for h in gc.get_referrers(_cache[0])])

_cache.clear()
print('После _cache.clear():', alive())

Живых Report: 100
Кто держит: ['list']
После _cache.clear(): 0


# 10. Сборка мусора в других реализациях Python

Подсчёт ссылок — особенность реализации, а не требование языка Python. Другие реализации могут искать живые объекты обходом графа от корней программы: всё недостижимое становится кандидатом на освобождение, включая циклы.

| Реализация | Как управляет памятью | Что происходит с недостижимым циклом | Когда освобождаются объекты |
| --- | --- | --- | --- |
| **CPython** | Подсчёт ссылок + отдельный cyclic GC | Его обнаруживает cyclic GC | Обычно сразу при обнулении счётчика; циклы — при сборке |
| **PyPy** | Трассирующий GC; стандартный `incminimark` использует поколения, перемещение объектов и инкрементальную сборку | Обрабатывается обычным GC, без отдельного механизма подсчёта ссылок | При сборке, не обязательно сразу после исчезновения последней ссылки |
| **Jython** | Использует GC виртуальной машины Java (JVM); конкретный алгоритм зависит от JVM и её настроек | Обрабатывается сборщиком JVM | Когда JVM выполнит соответствующую сборку |
| **IronPython** | Использует GC платформы .NET (CLR), который определяет достижимость объектов и использует поколения | Обрабатывается сборщиком .NET | Когда CLR выполнит соответствующую сборку |


# Итоги

1. В CPython переменные и контейнеры хранят ссылки на объекты.
2. Большинство объектов освобождается сразу, когда число сильных ссылок становится нулём.
3. Циклы не устраняются одним подсчётом ссылок; их периодически ищет cyclic GC.
4. Поколения делают сборку эффективнее: новые объекты проверяются чаще, старые — реже.
5. Обычно не нужно управлять GC вручную: сначала измерить проблему, затем искать удерживающие ссылки.
6. `weakref` подходит для не владеющих объектом связей, а `with` — для надёжного закрытия ресурсов.


## Источники

- [Документация модуля gc](https://docs.python.org/3/library/gc.html)
- [Документация tracemalloc](https://docs.python.org/3/library/tracemalloc.html)
- [Документация weakref](https://docs.python.org/3/library/weakref.html)
- [PEP 442: Safe object finalization](https://peps.python.org/pep-0442/)
- [Everything You Need to Know About How Python Manages Memory](https://machinelearningmastery.com/everything-you-need-to-know-about-how-python-manages-memory/)